# Working with datasets in multiple languages with ClassifAI and Multilingual Vectoriser Models

ClassifAI is a tool to help in the creation and serving of searchable vector databases, for text classification tasks. 

It has three core components:

1. Vectorisers - Models for converting text to vectors
2. Indexers - Classes for building VectorStores from text datasets, which you can search
3. Servers - Allow you to deploy VectorStores with a Rest-API interface


This notebook shows how to work with text data written in more than one language.

If your dataset contains, for example, both English and French text, you can build a ClassifAI `VectorStore` that can be searched in **either** language — and get relevant results back in **both** languages, regardless of which language you searched in.

This works because of **multilingual encoder models**. These are Vectoriser models that convert text from many different languages into a shared embedding space, so that text with the same meaning ends up with similar embeddings, no matter what language it's written in.

This notebook uses a multilingual model from HuggingFace, along with a dataset containing multiple languages, to demonstrate this in practice.

## In this Notebook...

We will show:

* The core workings of the Vectotiser Class and its responsibilities, <b>in a multilingual setting.</b>
* How to use multilingual encoding models from HuggingFace within ClassifAI using ClassifAI's `HuggingFaceVectoriser` class
* Examples of building a `VectorStore` knowledgebase containing english and non-english text, and examples of searching the knowledgebase with english and non-english queries.

## Vectorisers and Multilingual Encoding

![English_Vectoriser_Image](./files/vectoriser.png)


As seen above, a Vectorisers' sole responsibility is to convert text to a vector representation. Each Vectoriser class must implement a `transform()` method that will:

1. accept a string or list of N strings as an argument
2. return a numpy array of dimension [N,Y] where N matches the number of input strings, and Y is the embedding dimension)

By enforcing this, the Indexers and Servers modules can reliably work with any Vectoriser object to perform the various search/classification functions required by ClassifAI.

All a developer has to consider when building their own Vectoriser is the logic of this `transform()` method.

![Multilingual_Vectoriser_Image](./files/vectoriser_multilingual.png)

Some Vectoriser (embedding) models are trained to understand many languages at once. These are called **multilingual encoder models**.

As shown above, sentences in different languages go into the model, and come out as embeddings in a **shared embedding space**. If two sentences mean the same thing — even if they're written in different languages — their embeddings will be very similar.



In the context of ClassifAI, this means we can build a `VectorStore` from a CSV file containing text in multiple languages, and the resulting embeddings will still reflect the *meaning* of the text — not just its language.

In the diagram below, each dot represents a piece of text, coloured by language (black = English, blue = French, green = Italian). Dots that are **close together** represent sentences that mean similar things, even though they're written in different languages:

![Multilingual_Vectoriser_Image](./files/vectorstore_2d_vis_multilingual.png)

The `VectorStore` can be searched as normal, using the same functionality as any other ClassifAI use case — but because its using a multilingual Vectoriser, the search query can be written in **any language** the model supports, and it will still return relevant results regardless of the language of the original text.

## Example Implementation

### Vectoriser

This implememtatopm requires an appropriate embedding model that works in a multilingual fashion. For this demo we've chosen to use `Granite-Embedding-97M-Multilingual-R2` - an embedding model provided by IBM and available on HuggingFace that supports over 200 languages. For more information check out the model (and find other multilingual embedding models) on HuggingFace at: https://huggingface.co/ibm-granite/granite-embedding-97m-multilingual-r2


With ClassifAI, HuggingFace embedding models can be loaded with the Vectorisers module's `HuggingFaceVectoriser` class, the exact same way a monolingual embedding model would be loaded.

In [ ]:
from classifai.vectorisers import HuggingFaceVectoriser

multilingual_vectoriser = HuggingFaceVectoriser(model_name="ibm-granite/granite-embedding-97m-multilingual-r2")

The vectoriser's `transform()` method can be called that will convert text to embedding representation.

In [ ]:
embedding_from_english = multilingual_vectoriser.transform("ambulance driver")

embedding_from_french = multilingual_vectoriser.transform("conducteur d'ambulance")

embedding_from_italian = multilingual_vectoriser.transform("autista di ambulanza")

In [ ]:
print(embedding_from_english.shape)
print(embedding_from_french.shape)
print(embedding_from_italian)

### Dataset

For this example notebook, Generative AI was used to make an example dataset that contains fake SOC data, with text written in English and French. This dataset can be used, with the Granite-embedding model to build a ClassifAI VectorStore. Then this notebook will try out searching the `VectorStore` in a variety of languages (inlcuding languages other than English and French).

The below cell loads the CSV file into a Pandas dataframe, and displays the top 5 entries to showcase the content. It contains profession names with a short description of the work with a corresponding 'fake' SOC label. 


In [ ]:
import pandas as pd

multilingual_dataset = pd.read_csv("./data/fake_multilingual_soc_dataset.csv")
multilingual_dataset.head()

### Building the VectorStore

The process for creating a `VectorStore` is identical to the case of making a `VectorStore` for a CSV file of text all in the same language.

In [ ]:
from classifai.indexers import VectorStore

# this is standard code to construct a VectorStore from a CSV file of data and using an instantiated Vectoriser model.
demo_vectorstore = VectorStore(
    file_name="./data/fake_multilingual_soc_dataset.csv",
    data_type="csv",
    vectoriser=multilingual_vectoriser,
    skip_save=True,
)

### Searching with Enlgish queries

With the `VectorStore` object instantiated, now call the `search()` method - it can be seen that the below search query passed in English returns relevant results in multiple languages.

In [ ]:
from classifai.indexers.dataclasses import VectorStoreSearchInput

# creating a VectorStoreSearchInput object to pass to the search method
english_search_input = VectorStoreSearchInput({"id": [1], "query": ["medical doctor"]})

# calling the search method and displaying the results
demo_vectorstore.search(english_search_input, n_results=5)

The resulting output of the cell above shows that while there isn't a medical doctor profession listed in our dataset, the top few results are all medical related results from both French and English, despite the original query being only in English. 

### Searching with non-English Queries

In the cell below, an example query is written in French.

The english translation of the below query is <b>"A person who repairs cars and trucks"</b>

In [ ]:
# creating a VectorStoreSearchInput object to pass to the search method, this time with french
french_search_input = VectorStoreSearchInput(
    {"id": [1], "query": ["Une personne qui répare des voitures et des camions"]}
)

# calling the search method and displaying the results
demo_vectorstore.search(french_search_input, n_results=5)

Among the top results of the previous cell's output is the 'Mechanic" entry from the dataset which shows that the VectorStore is accepting a french query and returning an english result. The other top results are in french but are also seemingly relevant.

The VectorStore can also be searched in other languages that are not English or French, beacause as mentioned earlier the IBM Granite embedding model supports many languages.

Below the query "Una persona che progetta e testa programmi informatici e applicazioni" is passed to the VectorStore `search()` method - the English translation is: <b>"A person who designs and tests computer programs and applications"</b>

In [ ]:
# creating a VectorStoreSearchInput object to pass to the search method, this time with french
italian_search_input = VectorStoreSearchInput(
    {"id": [1], "query": ["Una persona che progetta e testa programmi informatici e applicazioni"]}
)

# calling the search method and displaying the results
demo_vectorstore.search(italian_search_input, n_results=5)

The above output shows several highly ranked results that are relevant from both French and English sources including "Ingénieur logiciel" (Software Engineer), "Software Developer" (which is an english result), and "Développeur web" (the French for Web Developer)

### Querying in many languages

Finally, its also possible to pass multiple queries to the search method at one time where the queries are in different languages. In the next cell, <b>the same query</b> is passed in English, French and Italian.


The results for each query should be similar, as the queries are translations of one another.

In [ ]:
# three queries that all mean the same thing in different languages
query_en = "A craftsman who builds and repairs wooden furniture and structures"
query_fr = "Un artisan qui construit et répare des meubles et des structures en bois"
query_it = "Un artigiano che costruisce e ripara mobili e strutture in legno"

# creating a input object for the search method.
search_input_multiple = VectorStoreSearchInput({"id": [1, 2, 3], "query": [query_en, query_fr, query_it]})

# searching, retrieving the top 3 candidates for each query.
demo_vectorstore.search(search_input_multiple, n_results=3)

While there is some difference in the top 3 results for each query, there is a large amount of overlap and generally the results are highly relevant throughout.

## Thats it!

* When choosing an embedding model from HuggingFace or from another service such as GCP, be sure to check which languages the embedding model supports. 
* Generally, monoligual models perform stronger on tasks for a specific single language than a multilingual model would. 
* Because these models are accessible through ClassifAI's `HuggingFaceVectoriser` class, the models are directly compatible with the other modules of the Package including the Servers module and Evaluation module,